# Capstone — Google Search Ranking & Discoverability
## Machine Learning for Content Refresh Prioritization

This notebook contains the complete research pipeline for **Lane 2: Content Refresh / Opportunity Scoring**.
It formulates the problem, audits the data, establishes an honest evaluation split on unseen clients, benchmarks machine learning models against a transparent rule baseline, inspects prediction errors, and outputs a ranked editorial action playbook.

*All analyses are reproducible and conducted strictly on safe, anonymized search intelligence data.*

## 1. Question

*The research question and the decision it supports.*

### Core Research Question
**"Which published content pages are at imminent risk of search traffic decline, and how can editorial teams systematically prioritize pages for a refresh to protect organic visibility?"**

### The Decision It Supports
Editorial teams face severe bandwidth constraints: an editorial desk can realistically refresh only 20 to 50 articles per week out of a catalog of thousands. Without a predictive prioritization model, editors rely on naive rules (e.g. "update whatever is older than 6 months") or gut feeling, spending hundreds of hours updating healthy pages while high-value ranking pages slip off Google's first page.

This work supports a concrete operational decision: **generating a weekly ranked refresh queue** where every flagged article carries an honest risk score, an action category, and a clear reason code.

In [1]:
# Setup environment and dependencies
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Environment initialized. Ready for analysis.")

Environment initialized. Ready for analysis.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Dataset Source & Warehouse Connection
This research is built on the **FlyRank Search Intelligence Warehouse Release** hosted on Hugging Face ([](https://huggingface.co/datasets/FlyRank/internship-warehouse)), comprising ~79M daily performance rows across 17 months.

Following the workflow taught in Starter Notebook 03, I connect **DuckDB natively over ** using an authenticated read token to query the remote warehouse Parquet partitions without needing to download gigabytes of raw daily data.

- **Tables Queried:**
  - : Enterprise client directory with historical access windows.
  - : Inventory of 500,000+ published content URLs and metadata.
  - : Partitioned daily GSC impressions, clicks, and rank positions.
- **Unit of Analysis:** One row = one content page for one client, measured over a trailing 90-day observation window with a 30-day forward outcome window.
- **Exclusions (Leakage & Privacy Protection):**
  - Forward outcome metrics (, ) are strictly excluded from features.
  - Client and content identifiers are excluded from model inputs to prevent memorizing brand-specific layout styles.
  - All client names, raw URLs, and search query strings remain pseudonymized.

In [2]:
# 1. Connect DuckDB to the Hugging Face Warehouse release
import duckdb
import os

# Retrieve HF read token from local environment / .env
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN and os.path.exists('../../.env'):
    with open('../../.env') as f:
        for line in f:
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip()

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    REL = 'hf://datasets/FlyRank/internship-warehouse'
    print('Connecting to Hugging Face Warehouse (FlyRank/internship-warehouse)...')
    dim_clients_count = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/dim_clients.parquet')").fetchone()[0]
    dim_content_count = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/dim_content.parquet')").fetchone()[0]
    march_rows = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')").fetchone()[0]
    print(f"  dim_clients:                {dim_clients_count:>10,} rows")
    print(f"  dim_content:                {dim_content_count:>10,} rows")
    print(f"  fact_daily (March 2026):    {march_rows:>10,} rows")

# 2. Load the Lane 2 aggregated feature dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)

print(f"\nLane 2 Aggregated Dataset: {len(df):,} pages across {df['client_id'].nunique()} clients")
print(f"Target class balance (Decline Rate): {df['is_declining'].mean() * 100:.2f}%")

Connecting to Hugging Face Warehouse (FlyRank/internship-warehouse)...


  dim_clients:                       104 rows
  dim_content:                   519,606 rows
  fact_daily (March 2026):     9,841,378 rows

Lane 2 Aggregated Dataset: 30,000 pages across 32 clients
Target class balance (Decline Rate): 54.21%


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### 1. Honest Split Design (Grouped by Client)
I use a **GroupShuffleSplit partitioned on ** with  and . 

A standard random split allows pages from the same client to appear in both training and test partitions. Because content within a client shares identical domain authority and layout structures, random splits produce inflated test metrics. Partitioning strictly by client tests whether the model generalizes to a **new, unseen client website**.

### 2. Baseline Rule Formulation
To prove that machine learning provides real value, I compare models against the heuristic rule developed in Week 4: flagging striking-distance pages (, positions 4-10) that have not been updated in over 90 days (), weighted by search impression volume.

### 3. Model Architectures
- **Logistic Regression:** A transparent linear baseline fitted using standard scaling and median imputation.
- **Random Forest Classifier:** An ensemble of 100 decision trees (max depth 10) capable of learning non-linear interactions between freshness, position, and search volume.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Compute baseline score across all rows
is_striking = (df['position_tier'] == 'striking').astype(int)
is_stale = (df['days_since_last_update'] >= 90).astype(int)
df['baseline_score'] = is_striking * is_stale * df['impressions_90d']

# Grouped split on client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

print(f"Train set: {len(df_train):,} pages ({df_train['client_id'].nunique()} clients)")
print(f"Test set:  {len(df_test):,} pages ({df_test['client_id'].nunique()} clients)")

# Define feature columns
numeric_features = ['impressions_90d', 'clicks_90d', 'avg_position_clean', 'days_since_last_update', 'word_count']
categorical_features = ['position_tier', 'content_type', 'main_intent']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Pipelines
lr_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(solver='liblinear', random_state=42))
]) 
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
]) 
# Train models
lr_pipeline.fit(df_train, df_train['is_declining'])
rf_pipeline.fit(df_train, df_train['is_declining'])
print("Model training completed on grouped split.")

Train set: 22,885 pages (24 clients)
Test set:  7,115 pages (8 clients)


Model training completed on grouped split.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
from sklearn.metrics import roc_auc_score, roc_curve

y_test = df_test['is_declining'].values
base_probs = df_test['baseline_score'].values
lr_probs = lr_pipeline.predict_proba(df_test)[:, 1]
rf_probs = rf_pipeline.predict_proba(df_test)[:, 1]
df_test['rf_prob'] = rf_probs

def eval_metrics(name, probs, y_true):
    auc = roc_auc_score(y_true, probs)
    order = np.argsort(-probs)
    p10 = y_true[order[:10]].mean()
    p50 = y_true[order[:50]].mean()
    p100 = y_true[order[:100]].mean()
    return {'Model': name, 'ROC-AUC': f"{auc:.4f}", 'Precision@10': f"{p10:.2f}", 'Precision@50': f"{p50:.2f}", 'Precision@100': f"{p100:.2f}"}

base_rate = y_test.mean()
results = [
    {'Model': 'Base Rate (Random)', 'ROC-AUC': '0.5000', 'Precision@10': f"{base_rate:.2f}", 'Precision@50': f"{base_rate:.2f}", 'Precision@100': f"{base_rate:.2f}"},
    eval_metrics('Baseline Rule', base_probs, y_test),
    eval_metrics('Logistic Regression', lr_probs, y_test),
    eval_metrics('Random Forest', rf_probs, y_test)
]

res_table = pd.DataFrame(results)
print("=== Performance Comparison on Unseen Clients ===")
print(res_table.to_string(index=False))

# Feature Importance Analysis
rf_clf = rf_pipeline.named_steps['clf']
ohe_cols = rf_pipeline.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_features = numeric_features + list(ohe_cols)
importances = rf_clf.feature_importances_
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)

print("\n=== Top 5 Most Important Features ===")
print(feat_imp.head(5).to_string())

=== Performance Comparison on Unseen Clients ===
              Model ROC-AUC Precision@10 Precision@50 Precision@100
 Base Rate (Random)  0.5000         0.52         0.52          0.52
      Baseline Rule  0.5021         0.30         0.50          0.56
Logistic Regression  0.5211         0.50         0.50          0.59
      Random Forest  0.5986         0.70         0.74          0.72

=== Top 5 Most Important Features ===
impressions_90d           0.311436
word_count                0.175082
avg_position_clean        0.137325
days_since_last_update    0.094092
clicks_90d                0.089992


/opt/homebrew/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 5. Limitations

*What this work cannot claim.*

1. **Observational Correlation vs. Causal Proof:** 
   The model predicts which pages correlate with future decline. It cannot guarantee that refreshing a page will mechanically restore traffic or reverse search loss. Factors outside content quality (such as competitor campaigns or Google core algorithm shifts) affect performance.
2. **Low-Volume Volatility:** 
   Pages with very few impressions (< 50 per quarter) suffer from high label noise. A drop from 2 clicks to 0 is recorded as a 100% collapse, triggering false positive flags on trivial traffic fluctuations.
3. **Client Scaling Asymmetry:** 
   Larger websites produce thousands of pages with different baseline CTR curves than small niche sites. Grouped splitting helps mitigate this, but domain scale remains an influential factor.
4. **Sparse Engagement Signals:** 
   GA4 engagement and session data are only available on a subset of clients in the warehouse, requiring the primary model to rely primarily on Search Console visibility metrics.

In [5]:
# Quantifying limitation: Traffic volume distribution and target variance
low_vol_mask = df['impressions_90d'] < 50
print(f"Pages with < 50 impressions: {low_vol_mask.sum():,} ({low_vol_mask.mean() * 100:.1f}% of total)")
print(f"Decline rate for low-volume pages:  {df.loc[low_vol_mask, 'is_declining'].mean() * 100:.1f}%")
print(f"Decline rate for high-volume pages: {df.loc[~low_vol_mask, 'is_declining'].mean() * 100:.1f}%")

Pages with < 50 impressions: 6,479 (21.6% of total)
Decline rate for low-volume pages:  34.2%
Decline rate for high-volume pages: 59.7%


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

I operationalize model outputs into a **4-tier Action Playbook** designed for editorial workflows:

1. **Priority Refresh ():** High predicted decline risk (>= 0.65) on pages with meaningful search visibility (>= 500 impressions). Action: assign immediate editorial overhaul.
2. **Striking Distance Optimization ():** Ranking on page 1 striking positions (ranks 4-10) with moderate risk (>= 0.50). Action: optimize meta tags and internal links to push toward top 3.
3. **Maintain / Stable ():** Low risk (< 0.35). Action: do not touch; avoid wasting editorial hours.
4. **Deprioritize ():** Very low volume (< 50 impressions). Action: remove from manual review queue.

In [6]:
def assign_action(row):
    if row['impressions_90d'] < 50:
        return 'Deprioritize (Low Volume)'
    elif row['rf_prob'] >= 0.65 and row['impressions_90d'] >= 500:
        return 'Priority Refresh'
    elif row['position_tier'] == 'striking' and row['rf_prob'] >= 0.50:
        return 'Striking Distance Optimization'
    elif row['rf_prob'] < 0.35:
        return 'Maintain / Do Not Touch'
    else:
        return 'Monitor'

df_test['recommended_action'] = df_test.apply(assign_action, axis=1)

print("=== Recommended Action Queue Breakdown (Test Set) ===")
print(df_test['recommended_action'].value_counts().to_string())

# Display top 5 priority refresh recommendations for editorial assignment
priority_queue = df_test[df_test['recommended_action'] == 'Priority Refresh'].sort_values('rf_prob', ascending=False)
print("\n=== Top 5 Priority Refresh Queue Sample ===")
print(priority_queue[['content_id', 'impressions_90d', 'clicks_90d', 'avg_position_clean', 'days_since_last_update', 'rf_prob']].head(5).to_string(index=False))

=== Recommended Action Queue Breakdown (Test Set) ===
recommended_action
Monitor                           2696
Deprioritize (Low Volume)         2090
Priority Refresh                  1437
Striking Distance Optimization     721
Maintain / Do Not Touch            171

=== Top 5 Priority Refresh Queue Sample ===
          content_id  impressions_90d  clicks_90d  avg_position_clean  days_since_last_update  rf_prob
content_f55fd2d8ed04             2237           2                 1.3                     104 0.901860
content_9e6c26757e7b             1135           0                 2.9                     104 0.868271
content_3672013e1d63             2488           5                42.3                     104 0.859117
content_1d0963b56227             3445           3                39.0                     104 0.857954
content_7e3d9c85959b             2576           1                35.0                     104 0.857750


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

I generate four high-resolution visualizations styled with the project's visual identity palette (Forest Green , Alabaster White , Charcoal ) and save them directly to  for the public research paper.

In [7]:
os.makedirs('../../docs/charts', exist_ok=True)

# 1. Model Comparison Chart
models = ['Baseline Rule', 'Logistic Reg.', 'Random Forest']
auc_scores = [0.502, 0.521, 0.599]
p50_scores = [0.500, 0.500, 0.740]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.2))
fig.patch.set_facecolor('#FAF9F6')
ax.set_facecolor('#FAF9F6')

r1 = ax.bar(x - width/2, auc_scores, width, label='ROC-AUC', color='#2A4B3C')
r2 = ax.bar(x + width/2, p50_scores, width, label='Precision@50', color='#8A9A86')
ax.axhline(base_rate, color='#888888', linestyle='--', linewidth=1, label=f'Base Rate ({base_rate * 100:.1f}%)')
ax.set_ylabel('Score', fontsize=11, color='#111111')
ax.set_title('Model Performance vs Baseline (Unseen Client Split)', fontsize=13, fontweight='bold', color='#111111')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10, color='#111111')
ax.legend(frameon=True, facecolor='#FAF9F6', edgecolor='#E5E5E0')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('../../docs/charts/model_comparison.png', dpi=150)
plt.close()

# 2. Top Feature Importances
feat_top8 = feat_imp.sort_values(ascending=True).tail(8)
fig, ax = plt.subplots(figsize=(7, 4.2))
fig.patch.set_facecolor('#FAF9F6')
ax.set_facecolor('#FAF9F6')
ax.barh(feat_top8.index, feat_top8.values, color='#2A4B3C', height=0.6)
ax.set_title('Top 8 Feature Importances (Random Forest)', fontsize=12, fontweight='bold', color='#111111')
ax.set_xlabel('Relative Importance', fontsize=10, color='#111111')
ax.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('../../docs/charts/feature_importance.png', dpi=150)
plt.close()

# 3. ROC Curves
fpr_base, tpr_base, _ = roc_curve(y_test, base_probs)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor('#FAF9F6')
ax.set_facecolor('#FAF9F6')
ax.plot(fpr_base, tpr_base, label='Baseline Rule (AUC = 0.502)', color='#888888', linestyle='--')
ax.plot(fpr_lr, tpr_lr, label='Logistic Reg. (AUC = 0.521)', color='#4A6B82', linestyle='-.')
ax.plot(fpr_rf, tpr_rf, label='Random Forest (AUC = 0.599)', color='#2A4B3C', linewidth=2)
ax.plot([0, 1], [0, 1], color='#CCCCCC', linestyle=':')
ax.set_xlabel('False Positive Rate', fontsize=10, color='#111111')
ax.set_ylabel('True Positive Rate', fontsize=10, color='#111111')
ax.set_title('ROC Curves Comparison on Unseen Clients', fontsize=12, fontweight='bold', color='#111111')
ax.legend(loc='lower right', facecolor='#FAF9F6', edgecolor='#E5E5E0')
ax.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('../../docs/charts/roc_curves.png', dpi=150)
plt.close()

# 4. Action Distribution
action_counts = df_test['recommended_action'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4.2))
fig.patch.set_facecolor('#FAF9F6')
ax.set_facecolor('#FAF9F6')
colors = ['#2A4B3C', '#5B7C6E', '#8A9A86', '#B8C4B5', '#D5DDD2']
ax.bar(action_counts.index, action_counts.values, color=colors[:len(action_counts)], width=0.55)
ax.set_title('Recommended Action Distribution (Unseen Test Client Queue)', fontsize=12, fontweight='bold', color='#111111')
ax.set_ylabel('Number of Pages', fontsize=10, color='#111111')
plt.xticks(rotation=20, ha='right', fontsize=9)
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('../../docs/charts/action_distribution.png', dpi=150)
plt.close()

print("All 4 publication charts successfully written to docs/charts/")

All 4 publication charts successfully written to docs/charts/


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under  — then submit your repo URL on the card. Done.